# CareLoop Clinics: No-Show Guard
## Part 1: Python & Data Foundations (ingest, validate, clean, enrich)

**Input:** `data/appointments.csv`, `data/clinics.csv` (originals, never edited).
**Outputs (hand-off files):** `rejected_rows.json`, `appointments_clean.csv`, `clinics_enriched.csv`, `load_data.sql`.

Conventions used throughout: percentages rounded to 1 decimal, `np.std` = population std (NumPy default), quartiles with the pandas default method.

### Setup
In Google Colab, upload the three original files into a folder called `data/` (Files pane → create folder `data` → upload). The cell below just checks they are there.

In [1]:
import os, csv, json
import numpy as np
import pandas as pd
import requests
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)

DATA = "data"
for f in ["appointments.csv", "clinics.csv", "followup_notes.csv"]:
    print(f, "found" if os.path.exists(os.path.join(DATA, f)) else "MISSING - upload it to data/")
print("pandas", pd.__version__, "| numpy", np.__version__)

appointments.csv found
clinics.csv found
followup_notes.csv found
pandas 2.2.3 | numpy 2.4.6


## Task 1.1: Read the file the foundations way (csv module only)
`parse_row` converts `age`, `fee_inr`, `prior_no_shows` to `int` and `distance_km` to `float`.
A row whose value cannot be converted (for example a blank cell) raises `ValueError` and goes to `rejected`.

In [2]:
with open(os.path.join(DATA, "appointments.csv"), newline="") as f:
    reader = csv.DictReader(f)
    header = reader.fieldnames
    raw_rows = list(reader)

print("Header:", header)
print("Row count:", len(raw_rows))

Header: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']
Row count: 1020


In [3]:
def parse_row(row):
    """Return a copy of the row with numeric columns converted; raises ValueError if any conversion fails."""
    out = dict(row)
    out["age"] = int(row["age"])
    out["fee_inr"] = int(row["fee_inr"])
    out["prior_no_shows"] = int(row["prior_no_shows"])
    out["distance_km"] = float(row["distance_km"])
    return out

accepted, rejected = [], []
for row in raw_rows:
    try:
        accepted.append(parse_row(row))
    except ValueError as e:
        rejected.append({"row": row, "error": str(e)})

print("Accepted:", len(accepted))
print("Rejected:", len(rejected))
blank_age = sum(1 for r in rejected if r["row"]["age"].strip() == "")
blank_dist = sum(1 for r in rejected if r["row"]["distance_km"].strip() == "")
print("  rejected with blank age:", blank_age)
print("  rejected with blank distance_km:", blank_dist)

with open("rejected_rows.json", "w") as f:
    json.dump(rejected, f, indent=2)
print("Saved rejected_rows.json")

Accepted: 988
Rejected: 32
  rejected with blank age: 12
  rejected with blank distance_km: 20
Saved rejected_rows.json


A type check is not a validity check: rows with impossible ages still pass `int()`, so they are *accepted* here.

In [4]:
impossible = [r for r in accepted if r["age"] < 1 or r["age"] > 100]
print("Accepted rows with impossible ages:", len(impossible), [r["age"] for r in impossible])

Accepted rows with impossible ages: 5 [0, 150, -4, 250, 199]


## Task 1.2: Inspect and clean, pass 1 (pandas, starting again from the original file)

In [5]:
df = pd.read_csv(os.path.join(DATA, "appointments.csv"))
print("Shape:", df.shape)
print()
df.info()
print()
print("Nulls per column:")
print(df.isnull().sum())
print()
print("Exact duplicate rows:", df.duplicated().sum())

Shape: (1020, 14)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   object 
 1   patient_id       1020 non-null   object 
 2   clinic_id        1020 non-null   object 
 3   booked_on        1020 non-null   object 
 4   appt_date        1019 non-null   object 
 5   age              1008 non-null   float64
 6   gender           1020 non-null   object 
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   object 
 10  visit_type       1020 non-null   object 
 11  reminder_sent    1020 non-null   object 
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   object 
dtypes: float64(2), int64(2), object(10)
memory usage: 111.7+ KB

Nulls per column:
appointment_id      0
patient_id          0
clinic_id 

Order of fixes: (a) exact duplicates → (b) channel labels → (c) dates → (d) drop rows with no usable date or status.

In [6]:
print("Rows at start:", len(df))

# (a) drop exact duplicate rows
df = df.drop_duplicates()
print("(a) after dropping exact duplicates:", len(df))

# (b) tidy booking_channel
print("    booking_channel distinct raw spellings:", df["booking_channel"].nunique(), sorted(df["booking_channel"].unique()))
df["booking_channel"] = df["booking_channel"].str.strip().str.title()
print("(b) after cleaning booking_channel:", len(df), "| distinct:", df["booking_channel"].nunique(), sorted(df["booking_channel"].unique()))

# (c) convert dates
before_na = df["appt_date"].isna().sum()
df["appt_date"] = pd.to_datetime(df["appt_date"], errors="coerce")
df["booked_on"] = pd.to_datetime(df["booked_on"])
print("(c) after converting dates:", len(df), "| appt_date missing:", df["appt_date"].isna().sum(),
      f"(already blank/N/A: {before_na}, could not be parsed: {df['appt_date'].isna().sum() - before_na})")

# (d) drop rows with a missing appt_date or status
print("    rows with missing appt_date:", df["appt_date"].isna().sum(), "| missing status:", df["status"].isna().sum())
df = df.dropna(subset=["appt_date", "status"])
print("(d) after dropping missing appt_date / status:", len(df))

Rows at start: 1020
(a) after dropping exact duplicates: 1000
    booking_channel distinct raw spellings: 16 [' App', ' phone', ' walk-in', 'APP ', 'App', 'App ', 'PHONE', 'Phone', 'Phone ', 'WALK-IN', 'Walk-In', 'Walk-in', 'Walk-in ', 'app', 'phone', 'walk-in']
(b) after cleaning booking_channel: 1000 | distinct: 3 ['App', 'Phone', 'Walk-In']
(c) after converting dates: 1000 | appt_date missing: 6 (already blank/N/A: 1, could not be parsed: 5)
    rows with missing appt_date: 6 | missing status: 4
(d) after dropping missing appt_date / status: 990


## Task 1.3: Clean, pass 2 (validity, gaps, outliers)
Order matters: invalid → missing, then fill gaps, then compute quartiles/fences on the filled columns.

In [7]:
# (a) impossible ages -> missing
invalid_age = (df["age"] < 1) | (df["age"] > 100)
print("(a) invalid ages:", invalid_age.sum(), sorted(df.loc[invalid_age, "age"].tolist()))
df.loc[invalid_age, "age"] = np.nan
print("    missing ages now:", df["age"].isna().sum())

# (b) fill gaps with the median
age_median = df["age"].median()
dist_median = df["distance_km"].median()
print("(b) median age:", age_median, "| median distance_km:", dist_median)
df["age"] = df["age"].fillna(age_median).round().astype(int)
df["distance_km"] = df["distance_km"].fillna(dist_median)

# (c) IQR capping
for col in ["distance_km", "fee_inr"]:
    q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_high = (df[col] > high).sum()
    n_low = (df[col] < low).sum()
    df[col] = np.clip(df[col], low, high)
    print(f"(c) {col}: Q1={q1}, Q3={q3}, IQR={iqr}, lower fence={low}, upper fence={high}")
    print(f"    capped {n_high + n_low} values ({n_high} high, {n_low} low)")

# (d) rounding
df["distance_km"] = df["distance_km"].round(1)
df["fee_inr"] = df["fee_inr"].round().astype(int)
print("(d) largest fee after capping:", df["fee_inr"].max(), "| largest distance:", df["distance_km"].max())
print("Remaining nulls:", int(df.isnull().sum().sum()))

(a) invalid ages: 5 [-4.0, 0.0, 150.0, 199.0, 250.0]
    missing ages now: 17
(b) median age: 48.0 | median distance_km: 8.25
(c) distance_km: Q1=4.625, Q3=13.0, IQR=8.375, lower fence=-7.9375, upper fence=25.5625
    capped 26 values (26 high, 0 low)
(c) fee_inr: Q1=500.0, Q3=787.5, IQR=287.5, lower fence=68.75, upper fence=1218.75
    capped 3 values (3 high, 0 low)
(d) largest fee after capping: 1219 | largest distance: 25.6
Remaining nulls: 0


## Task 1.4: Features and a NumPy summary (no loops)

In [8]:
df["lead_days"] = (df["appt_date"] - df["booked_on"]).dt.days
df["no_show"] = (df["status"] == "No-Show").astype(int)

print("lead_days range:", df["lead_days"].min(), "to", df["lead_days"].max())
print(f"No-shows: {df['no_show'].sum()} of {len(df)} ({round(100 * df['no_show'].mean(), 1)}%)")

X = df[["age", "distance_km", "fee_inr", "lead_days"]].to_numpy()

# (a) column means
means = X.mean(axis=0)
print("(a) column means [age, distance_km, fee_inr, lead_days]:", np.round(means, 2))

# (b) boolean mask for long lead times
mask = X[:, 3] > 14
print(f"(b) rows with lead_days > 14: {mask.sum()} ({round(100 * mask.mean(), 1)}%)")

# (c) z-scores of distance_km
x = X[:, 1]
z = (x - np.mean(x)) / np.std(x)
print(f"(c) distance_km: mean={np.mean(x):.3f}, std={np.std(x):.3f}, rows with |z| > 2: {(np.abs(z) > 2).sum()}")

lead_days range: 1 to 45
No-shows: 218 of 990 (22.0%)
(a) column means [age, distance_km, fee_inr, lead_days]: [ 48.17   9.5  635.71  11.82]
(b) rows with lead_days > 14: 296 (29.9%)
(c) distance_km: mean=9.501, std=6.157, rows with |z| > 2: 53


## Task 1.5: Enrich the clinics from a live API (Open-Meteo, no key needed)
`temp_c` is the **first** hourly `temperature_2m` value (the midnight-UTC reading, a stand-in rather than the current weather).
Each call has a 10-second timeout and one retry. If a clinic still fails, `temp_c` stays `None` and the reason is printed. No value is ever invented.

In [9]:
clinics = pd.read_csv(os.path.join(DATA, "clinics.csv"))
NAME_COL = "clinic_name"
print(clinics)

def fetch_temp(lat, lon):
    """Return (temp_c or None, status_code or None, reason)."""
    r = requests.get("https://api.open-meteo.com/v1/forecast",
                     params={"latitude": lat, "longitude": lon,
                             "hourly": "temperature_2m", "forecast_days": 1},
                     timeout=10)
    if r.status_code != 200:
        return None, r.status_code, f"status {r.status_code}"
    data = r.json()
    if "hourly" not in data or not data["hourly"].get("temperature_2m"):
        return None, r.status_code, "hourly temperature_2m missing in reply"
    return data["hourly"]["temperature_2m"][0], r.status_code, "ok"

temps = []
for _, c in clinics.iterrows():
    temp, reason = None, ""
    for attempt in (1, 2):                      # first try + one retry
        try:
            temp, code, reason = fetch_temp(c["latitude"], c["longitude"])
            print(f"{c[NAME_COL]}: attempt {attempt}, status {code}, temp_c={temp}")
            if temp is not None:
                break
        except requests.exceptions.RequestException as e:
            reason = f"{type(e).__name__}: {e}"
            print(f"{c[NAME_COL]}: attempt {attempt} failed ({type(e).__name__})")
    if temp is None:
        print(f"  -> FAILED for {c[NAME_COL]}: {reason[:150]}. temp_c stored as None.")
    temps.append(temp)

clinics["temp_c"] = temps
clinics.to_csv("clinics_enriched.csv", index=False)
print()
print(clinics)
print("Saved clinics_enriched.csv:", clinics.shape)

  clinic_id               clinic_name     city    tier  latitude  longitude
0       C01     Kothrud Family Clinic     Pune   Metro   18.5074    73.8077
1       C02      Hadapsar Care Centre     Pune   Metro   18.5089    73.9260
2       C03    Sitabuldi Health Point   Nagpur  Tier-2   21.1458    79.0882
3       C04      Malviya Nagar Clinic   Jaipur  Tier-2   26.8553    75.8150
4       C05     Vaishali Nagar Clinic   Jaipur  Tier-2   26.9124    75.7873
5       C06  Edappally Medical Centre    Kochi  Tier-2   10.0261    76.3085
6       C07        Vijay Nagar Clinic   Indore  Tier-2   22.7533    75.8937
7       C08    Gomti Nagar Care Point  Lucknow  Tier-2   26.8467    80.9462
Kothrud Family Clinic: attempt 1, status 200, temp_c=14.4
Hadapsar Care Centre: attempt 1, status 200, temp_c=15.7
Sitabuldi Health Point: attempt 1, status 200, temp_c=22.3
Malviya Nagar Clinic: attempt 1, status 200, temp_c=15.3
Vaishali Nagar Clinic: attempt 1, status 200, temp_c=15.3
Edappally Medical Centre: a

## Task 1.6: Export the clean table and write the SQL load script

In [10]:
COLS = ["appointment_id", "patient_id", "clinic_id", "booked_on", "appt_date", "age",
        "gender", "distance_km", "fee_inr", "booking_channel", "visit_type",
        "reminder_sent", "prior_no_shows", "status", "lead_days", "no_show"]

out = df.sort_values("appointment_id").copy()
out["booked_on"] = out["booked_on"].dt.strftime("%Y-%m-%d")
out["appt_date"] = out["appt_date"].dt.strftime("%Y-%m-%d")
out = out[COLS]

print("appointment_id unique:", out["appointment_id"].is_unique, "| distinct values:", out["appointment_id"].nunique())
print("Missing values per column (all must be 0):")
print(out.isnull().sum())
assert out["appointment_id"].is_unique and out.isnull().sum().sum() == 0

out.to_csv("appointments_clean.csv", index=False)
print("Saved appointments_clean.csv:", out.shape)

appointment_id unique: True | distinct values: 990
Missing values per column (all must be 0):
appointment_id     0
patient_id         0
clinic_id          0
booked_on          0
appt_date          0
age                0
gender             0
distance_km        0
fee_inr            0
booking_channel    0
visit_type         0
reminder_sent      0
prior_no_shows     0
status             0
lead_days          0
no_show            0
dtype: int64
Saved appointments_clean.csv: (990, 16)


In [11]:
def sql_value(v):
    """Text in single quotes, numbers unquoted, missing values as NULL."""
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return "NULL"
    if isinstance(v, str):
        return f"'{v}'"
    return f"{v}"

lines = []
# clinics first: appointments.clinic_id is a foreign key to clinics
for row in clinics.itertuples(index=False):
    lines.append(f"INSERT INTO clinics VALUES ({','.join(sql_value(v) for v in row)});")
for row in out.itertuples(index=False):
    lines.append(f"INSERT INTO appointments VALUES ({','.join(sql_value(v) for v in row)});")

with open("load_data.sql", "w") as f:
    f.write("\n".join(lines) + "\n")

n_cl = sum(l.startswith("INSERT INTO clinics") for l in lines)
n_ap = sum(l.startswith("INSERT INTO appointments") for l in lines)
print(f"load_data.sql: {n_cl} clinics + {n_ap} appointments = {len(lines)} INSERT statements")
print(lines[0]); print(lines[n_cl])

load_data.sql: 8 clinics + 990 appointments = 998 INSERT statements
INSERT INTO clinics VALUES ('C01','Kothrud Family Clinic','Pune','Metro',18.5074,73.8077,14.4);
INSERT INTO appointments VALUES ('A0001','P509','C05','2025-01-24','2025-01-25',60,'F',18.0,850,'Phone','Consultation','Yes',1,'No-Show',1,1);


## Task 2.2 cross-check in Python
The clinic-level counts from the SQL query in `02_queries.sql`, reproduced from the exported files with `groupby`. They must match the SQL output exactly.

In [12]:
chk = pd.read_csv("appointments_clean.csv").merge(pd.read_csv("clinics_enriched.csv"), on="clinic_id")
by_clinic = (chk.groupby(["clinic_name", "city"])
                .agg(appointments=("no_show", "size"), no_shows=("no_show", "sum"))
                .reset_index())
by_clinic["no_show_pct"] = (100 * by_clinic["no_shows"] / by_clinic["appointments"]).round(1)
print(by_clinic.sort_values("no_show_pct", ascending=False).to_string(index=False))
print("Total:", by_clinic["appointments"].sum(), "appointments,", by_clinic["no_shows"].sum(), "no-shows")

             clinic_name    city  appointments  no_shows  no_show_pct
  Gomti Nagar Care Point Lucknow            90        30         33.3
    Hadapsar Care Centre    Pune           143        39         27.3
    Malviya Nagar Clinic  Jaipur           142        38         26.8
   Kothrud Family Clinic    Pune           160        36         22.5
      Vijay Nagar Clinic  Indore           101        21         20.8
Edappally Medical Centre   Kochi           142        23         16.2
  Sitabuldi Health Point  Nagpur            82        13         15.9
   Vaishali Nagar Clinic  Jaipur           130        18         13.8
Total: 990 appointments, 218 no-shows


### Part 1 summary
The raw export of 1,020 rows becomes a clean table of 990 unique appointments with no missing values. See the printed outputs above for every count (duplicates, unreadable dates, invalid ages, capped outliers).